JobHop is a career trajectory dataset, so the cleanest formulation is:
- CV = a person’s past experiences (sequence of ESCO occupations)
- JD = the next occupation they moved into (represented using your ESCO JD text from jd_df)

This becomes a ranking problem:
- Given a CV history, rank candidate job descriptions; the true next job should rank highly.

<a class="anchor" id="chapter1"></a>

# 1. Imports

</a>

In [2]:
from datasets import load_dataset

/opt/anaconda3/envs/ResumeMatcherThesis/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
import pandas as pd
import numpy as np

<a class="anchor" id="chapter2"></a>

# 2. Load Data

</a>

In [14]:
ds = load_dataset("aida-ugent/JobHop")

In [15]:
ds

DatasetDict({
    train: Dataset({
        features: ['person_id', 'matched_label', 'matched_description', 'matched_code', 'start_date', 'end_date', 'university_studies'],
        num_rows: 1341832
    })
    validation: Dataset({
        features: ['person_id', 'matched_label', 'matched_description', 'matched_code', 'start_date', 'end_date', 'university_studies'],
        num_rows: 167945
    })
    test: Dataset({
        features: ['person_id', 'matched_label', 'matched_description', 'matched_code', 'start_date', 'end_date', 'university_studies'],
        num_rows: 167924
    })
})

In [16]:
train_ds = ds["train"]
val_ds   = ds["validation"]
test_ds  = ds["test"]

In [19]:
train_df = train_ds.to_pandas()
val_df   = val_ds.to_pandas()
test_df  = test_ds.to_pandas()

<a class="anchor" id="chapter3"></a>

# 3. Initial Analysis

</a>

In [20]:
train_df.head()

,person_id,matched_label,matched_description,matched_code,start_date,end_date,university_studies
0,0,resource manager,Resource managers manage resources for all pot...,1324.8.3,Q1 2016,Q2 2019,True
1,0,health and safety officer,Health and safety officers execute plans for t...,2263.3,Q1 2017,Q2 2019,True
2,0,integration engineer,Integration engineers develop and implement so...,2511.17,Q1 2013,Q1 2016,True
3,0,programme manager,Programme managers coordinate and oversee seve...,1213.4,Q2 2012,Q1 2013,True
4,0,product development engineering drafter,Product development engineering drafters desig...,3118.3.12,Q1 2011,Q2 2012,True


In [21]:
cv_train = (
    train_df
    .groupby("person_id")
    .agg({
        "matched_code": list,
        "matched_label": list,
        "matched_description": list,
        "university_studies": "max"
    })
    .reset_index()
)

In [22]:
cv_train.head()

,person_id,matched_code,matched_label,matched_description,university_studies
0,0,"[1324.8.3, 2263.3, 2511.17, 1213.4, 3118.3.12,...","[resource manager, health and safety officer, ...",[Resource managers manage resources for all po...,True
1,1,"[2512.2, 2512.2, 2153.1.1, unknown, 5414.1.9, ...","[software analyst, software analyst, telecommu...",[Software analysts elicit and prioritise user ...,True
2,2,"[2411.1, 2411.1, 2433.6.6, 4110.1]","[accountant, accountant, technical sales repre...",[Accountants review and analyse financial stat...,True
3,4,[2611.1],[lawyer],[Lawyers provide legal advice to clients and a...,False
4,8,"[unknown, unknown, unknown, unknown]","[unknown, unknown, unknown, unknown]","[unknown, unknown, unknown, unknown]",False


In [23]:
len(cv_train)

288965

Each résumé is constructed by aggregating all job experiences associated with a unique person identifier, yielding a structured CV representation grounded in ESCO occupation descriptions.